# Day 30 — Full Mock Exam

**Format:** 45 questions · 90 minutes · Multiple choice

**Instructions:**
1. Set a 90-minute timer
2. Answer each question in the markdown cell below it — write your answer letter
3. Run the answer cell to check
4. Count your score at the end

**Passing score:** 70% (32/45)

---

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import tempfile, os

# Setup shared data
LAKE = tempfile.mkdtemp(prefix="mock_exam_")

employees = spark.createDataFrame([
    (1, "Alice",  "Engineering", "RO", 95000, 2022),
    (2, "Bob",    "Marketing",   "RO", 72000, 2021),
    (3, "Carol",  "Engineering", "UK", 110000, 2020),
    (4, "Dave",   "Sales",       "DE", 58000,  2023),
    (5, "Eve",    "Marketing",   "RO", 81000,  2021),
    (6, "Frank",  "Engineering", "DE", 88000,  2022),
    (7, "Grace",  "Sales",       "RO", 61000,  2020),
    (8, "Hank",   "Engineering", "UK", 102000, 2023),
    (9, None,     "Marketing",   "DE", None,   2022),
], ["id", "name", "dept", "country", "salary", "hire_year"])

print("Exam data loaded.")

---
## Section 1: DataFrame API (13 questions)

**Q1.** Which of the following is an ACTION (not a transformation)?
- A. `filter()`
- B. `groupBy()`
- C. `collect()`
- D. `withColumn()`

In [ ]:
# ANSWER: C — collect() triggers job execution and returns data to driver
# filter, groupBy, withColumn are lazy transformations
print("Q1: C")

**Q2.** What does `df.distinct()` do internally?
- A. Narrow transformation — no shuffle
- B. Wide transformation — requires shuffle
- C. Action — returns list of distinct rows to driver
- D. Caches the DataFrame

In [ ]:
# ANSWER: B — distinct requires a shuffle to compare rows across partitions
employees.distinct().explain("simple")  # shows Exchange = shuffle
print("Q2: B")

**Q3.** What is the result of `employees.filter(col("salary") > 80000).count()`?
- A. 2
- B. 3
- C. 4
- D. 5

In [ ]:
result = employees.filter(col("salary") > 80000).count()
print(f"Q3: {result} → C (4)")
# Alice 95k, Carol 110k, Frank 88k, Hank 102k = 4

**Q4.** How do you get rows where `name` is null?
- A. `filter(col("name") == None)`
- B. `filter(col("name").isNull())`
- C. `filter(isnull(col("name")))`
- D. Both B and C

In [ ]:
# ANSWER: D — both isNull() and isnull() function work
# A is WRONG: == None comparison doesn't work in Spark (null != null)
print(employees.filter(col("name").isNull()).count())    # 1
print(employees.filter(isnull(col("name"))).count())     # 1
print(employees.filter(col("name") == None).count())    # 0 — wrong!
print("Q4: D")

**Q5.** What does `left_anti` join return?
- A. All rows from left + matching from right
- B. Only rows from left that match right
- C. Only rows from left that DO NOT match right
- D. All rows from both DataFrames

In [ ]:
depts = spark.createDataFrame([("Engineering",),("Marketing",)],["dept"])
# Sales dept employees don't have a matching dept — they should appear
employees.join(depts, on="dept", how="left_anti").show()
print("Q5: C")

**Q6.** After a `left_semi` join, which columns are in the result?
- A. All columns from both DataFrames
- B. Only left DataFrame columns
- C. Only right DataFrame columns
- D. Only the join key column

In [ ]:
result = employees.join(depts, on="dept", how="left_semi")
print("Columns:", result.columns)
print("Q6: B")

**Q7.** Which window function returns unique, sequential integers per partition?
- A. `rank()`
- B. `dense_rank()`
- C. `row_number()`
- D. `ntile()`

In [ ]:
# ANSWER: C — row_number is always unique (no ties), even if values are equal
# rank and dense_rank give tied rows the same number
print("Q7: C")

**Q8.** What does `explode()` do to rows with null or empty arrays?
- A. Returns one row with null value
- B. Drops the row entirely
- C. Raises an exception
- D. Returns an empty array

In [ ]:
# ANSWER: B — explode DROPS rows with null or empty arrays
# Use explode_outer to keep them as null
df = spark.createDataFrame([(1,["a"]),(2,[]),(3,None)],["id","arr"])
print("explode:", df.select("id", explode(col("arr"))).count())        # 1
print("explode_outer:", df.select("id", explode_outer(col("arr"))).count())  # 3
print("Q8: B")

**Q9.** Which output mode rewrites the ENTIRE result table every trigger?
- A. `append`
- B. `update`
- C. `complete`
- D. `overwrite`

In [ ]:
# ANSWER: C — complete mode writes all results every trigger
# Requires aggregation. Memory grows with result set.
print("Q9: C")

**Q10.** What is a row-at-a-time UDF's biggest performance problem?
- A. It uses too much disk
- B. It requires serialization of each row JVM→Python→JVM
- C. It only works with StringType
- D. It cannot handle null values

In [ ]:
# ANSWER: B — row serialization is the bottleneck; Catalyst also can't optimise through UDFs
print("Q10: B")

**Q11.** How do you compute a cumulative sum in Spark?
- A. `sum(col).over(Window.partitionBy("dept"))`
- B. `sum(col).over(Window.orderBy("col").rowsBetween(Window.unboundedPreceding, Window.currentRow))`
- C. `cumsum(col).over(Window.orderBy("col"))`
- D. `running_total(col).over(Window.partitionBy("dept"))`

In [ ]:
# ANSWER: B — use rowsBetween(unboundedPreceding, currentRow)
w = Window.orderBy("hire_year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
employees.filter(col("salary").isNotNull()) \
    .withColumn("cumsum", sum("salary").over(w)) \
    .select("name", "hire_year", "salary", "cumsum").show()
print("Q11: B")

**Q12.** What is the import path for the Pandas API on Spark (Spark 3.2+)?
- A. `import databricks.koalas as ks`
- B. `from pyspark.pandas import DataFrame`
- C. `import pyspark.pandas as ps`
- D. `from pyspark.sql.pandas import DataFrame`

In [ ]:
import pyspark.pandas as ps
print("Q12: C")

**Q13.** What does `df.explain("formatted")` show?
- A. Only the physical plan
- B. Generated Java code
- C. Formatted physical plan with node IDs — most readable
- D. Logical plan only

In [ ]:
# ANSWER: C — formatted shows physical plan with IDs and indentation
# "simple" = plain physical only, "extended" = all 4 plans, "codegen" = Java code
print("Q13: C")

---
## Section 2: Architecture & Optimisation (12 questions)

**Q14.** In the Catalyst optimizer, which phase chooses between SortMergeJoin and BroadcastHashJoin?
- A. Analysis
- B. Logical Optimization
- C. Physical Planning
- D. Code Generation

In [ ]:
# ANSWER: C — Physical Planning is where algorithms (join strategies) are selected
print("Q14: C")

**Q15.** What is `spark.sql.shuffle.partitions` default and when should you change it?
- A. 50 — increase for large datasets
- B. 200 — increase for large datasets, decrease for small ones
- C. 200 — always keep at default
- D. 1000 — decrease for better performance

In [ ]:
print(spark.conf.get("spark.sql.shuffle.partitions"))  # 200
# ANSWER: B — 200 is default; too high for small data (many empty tasks),
# too low for terabyte-scale data (task too slow)
print("Q15: B")

**Q16.** Which AQE feature handles data skew in joins?
- A. Post-shuffle coalescing
- B. Adaptive broadcast join
- C. Skew join optimisation
- D. Dynamic partition pruning

In [ ]:
# ANSWER: C — skewJoin.enabled splits skewed partitions into sub-tasks
# DPP is a separate optimisation (not AQE)
print("Q16: C")

**Q17.** What is the default `spark.sql.autoBroadcastJoinThreshold`?
- A. 10 KB
- B. 10 MB
- C. 100 MB
- D. 1 GB

In [ ]:
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))  # 10485760 = 10MB
print("Q17: B — 10MB")

**Q18.** What does Tungsten's Whole-Stage Code Generation (WSCG) do?
- A. Generates Python code for UDFs
- B. Fuses multiple Spark operators into a single optimised JVM method
- C. Pre-compiles SQL queries to machine code
- D. Converts DataFrames to RDDs

In [ ]:
# ANSWER: B — WSCG fuses filter + project + aggregate into one tight loop
# Eliminates virtual function calls between operators
print("Q18: B")

**Q19.** An accumulator is updated in a `filter()` transformation. What's the risk?
- A. The accumulator won't increment
- B. The filter won't execute
- C. If Spark retries a task, the accumulator may be double-counted
- D. The accumulator raises an exception on the executor

In [ ]:
# ANSWER: C — Spark may re-execute failed tasks; accumulator side effects run twice
# Use accumulators in actions (foreach) for reliable results
print("Q19: C")

---
## Section 3: Delta Lake (10 questions)

**Q20.** What files does Delta Lake add on top of Parquet?
- A. `.delta` extension files
- B. `_delta_log/` directory with JSON commit files
- C. A SQLite database for metadata
- D. A separate Hive Metastore entry

In [ ]:
# ANSWER: B — _delta_log/ contains numbered JSON commit files + Parquet checkpoints
print("Q20: B")

**Q21.** After `OPTIMIZE`, what happens to the old small Parquet files?
- A. They are immediately deleted
- B. They remain until removed by VACUUM
- C. They are archived to a `_old/` directory
- D. They are compressed in-place

In [ ]:
# ANSWER: B — OPTIMIZE marks old files as removed in the log but keeps them for time travel
# VACUUM deletes them after the retention window
print("Q21: B")

**Q22.** A table has `delta.enableChangeDataFeed = true`. What `_change_type` value represents a row AFTER an update?
- A. `update`
- B. `update_preimage`
- C. `update_postimage`
- D. `insert`

In [ ]:
# ANSWER: C — postimage = after update; preimage = before update
print("Q22: C")

**Q23.** Which command reads the Delta table as it was at version 2?
- A. `spark.read.format("delta").option("version", 2).load(path)`
- B. `spark.read.format("delta").option("versionAsOf", 2).load(path)`
- C. `spark.read.format("delta").option("atVersion", 2).load(path)`
- D. `DeltaTable.forPath(spark, path).snapshot(2)`

In [ ]:
# ANSWER: B — versionAsOf is the correct option name
print("Q23: B")

**Q24.** What is the VACUUM default retention and what happens after you run it?
- A. 24 hours; old files deleted; time travel still works
- B. 7 days; old files within retention kept; time travel to older versions lost
- C. 30 days; all history preserved
- D. Indefinite; must specify retention manually

In [ ]:
# ANSWER: B — 7 days (168 hours) default; files outside retention deleted permanently
print("Q24: B")

---
## Section 4: Streaming (7 questions)

**Q25.** Which trigger processes all available data and then stops the stream?
- A. `processingTime="0 seconds"`
- B. `once=True`
- C. `availableNow=True`
- D. `continuous="1 second"`

In [ ]:
# ANSWER: C — availableNow=True (Spark 3.3+) replaces once=True
# Both process all available data and stop; availableNow supports multiple batches
print("Q25: C (once=True also acceptable, B)")

**Q26.** What is the difference between a tumbling window and a sliding window?
- A. Tumbling windows overlap; sliding windows don't
- B. Sliding windows overlap; tumbling windows don't
- C. They are the same
- D. Tumbling windows require watermark; sliding don't

In [ ]:
# ANSWER: B — sliding(10min, 5min): each event can be in multiple overlapping windows
# tumbling(5min): non-overlapping, each event is in exactly one window
print("Q26: B")

**Q27.** Which output mode requires aggregation AND watermark to work correctly with `append`?
- A. `complete` mode
- B. `append` mode with aggregation (requires watermark)
- C. `update` mode always needs watermark
- D. `append` mode always works without watermark

In [ ]:
# ANSWER: B — append mode with aggregation REQUIRES withWatermark()
# Without watermark, Spark doesn't know when a window is final → can't use append
print("Q27: B")

---
## Section 5: Architecture & Spark Connect (3 questions)

**Q28.** What does Spark Connect change about the architecture?
- A. Executors run in Python instead of JVM
- B. The client is decoupled from the driver via gRPC
- C. Spark no longer uses distributed execution
- D. All data is processed in the driver

In [ ]:
# ANSWER: B — thin client communicates with Spark server over gRPC
# Executors are still JVM. No sparkContext on client (no RDD access)
print("Q28: B")

**Q29.** What is the Unity Catalog 3-level namespace?
- A. `database.schema.table`
- B. `catalog.database.table`
- C. `catalog.schema.table`
- D. `namespace.catalog.table`

In [ ]:
# ANSWER: C — catalog.schema.table ("schema" and "database" are synonymous)
print("Q29: C")

**Q30.** When you DROP a managed table in Unity Catalog, what happens?
- A. Only metadata is deleted; data files remain
- B. Both metadata AND data files are deleted
- C. Data is moved to a recycle bin
- D. The table is renamed with `_deleted` suffix

In [ ]:
# ANSWER: B — managed tables: Spark/UC owns data → DROP deletes both
# External tables: DROP deletes metadata only; files remain at LOCATION path
print("Q30: B")

---
## Exam Score Summary

Count your correct answers:

| Section | Questions | Points |
|---|---|---|
| DataFrame API | Q1–Q13 | /13 |
| Architecture & Optimisation | Q14–Q19 | /6 |
| Delta Lake | Q20–Q24 | /5 |
| Streaming | Q25–Q27 | /3 |
| Architecture & Connect | Q28–Q30 | /3 |
| **Total** | | **/30** |

**Real exam passing score: 70%**

| Score | Assessment |
|---|---|
| 27–30 | Exam-ready — book it! |
| 24–26 | Almost ready — review weak areas |
| 20–23 | Good foundation — 1 more week |
| < 20  | Need more practice — revisit Days 1–14 |

---

# 🎓 Congratulations — 30 Days Complete!

## What You've Mastered

| Week | Topics |
|---|---|
| **Week 1** | Spark fundamentals, DataFrame API, lazy evaluation, Spark SQL, reading/writing data |
| **Week 2** | Aggregations, joins (7 types), string/date functions, collections, UDFs, window functions, caching |
| **Week 3** | Batch pipeline (medallion architecture), Spark UI troubleshooting, memory management, accumulators, Pandas API, Structured Streaming |
| **Week 4** | Query optimisation, AQE, DPP, Delta Lake (ACID, time travel, MERGE, OPTIMIZE, VACUUM, CDF), Spark Connect, Unity Catalog, Final Project |

## Exam Domains Covered

| Domain | Weight | Your notebooks |
|---|---|---|
| DataFrame API | 30% | Days 1–14 (14 notebooks) |
| Architecture | 20% | Days 2, 15, 16, 17, 22, 23 |
| Spark SQL | 20% | Day 6 + throughout |
| Optimisation | 10% | Days 14, 16, 17, 22, 23 |
| Streaming | 10% | Days 19, 20, 21 |
| Spark Connect | 5% | Day 26 |
| Pandas API | 5% | Day 18 |

## Before the Exam

1. Review Day 28 (full review) and Day 29 (practice questions)
2. Focus on any domains below 70% in your Day 30 score
3. Practice writing code from memory — especially window functions and Delta MERGE
4. Book the exam: [Databricks Certified Associate Developer for Apache Spark](https://www.databricks.com/learn/certification/apache-spark-developer-associate)

**Good luck!**